In [1]:
import torch
import pytorch_lightning as pl
from torch import nn
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from os.path import join
from sklearn.metrics import accuracy_score
from torch.optim import SGD
import numpy as np
from pytorch_lightning.loggers import TensorBoardLogger
import pytorch_lightning as pl
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision import transforms

In [2]:
NUM_EPOCHS = 150

In [3]:
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision import transforms

transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.1307,), (0.3081,)),
                                ]) #flatten serve per convertire le immagini in vettori di 784 unità 

mnist_train = MNIST(root='mnist',train=True, download=True, transform=transform)
mnist_test = MNIST(root='mnist',train=False, download=True, transform=transform)


mnist_train_loader = DataLoader(mnist_train, batch_size=1024, num_workers=2, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=1024, num_workers=2)

In [4]:
from torchvision.utils import make_grid

class conv_autoencoder(pl.LightningModule):
    def __init__(self):
        super(conv_autoencoder, self).__init__() 

        self.training_step_outputs = []   # save outputs in each batch to compute metric overall epoch
        self.training_step_targets = []   # save targets in each batch to compute metric overall epoch
        self.val_step_outputs = []        # save outputs in each batch to compute metric overall epoch
        self.val_step_targets = [] 
        
        self.encoder = nn.Sequential(nn.Conv2d(1,16,3, padding=1),
                                     nn.AvgPool2d(2),
                                     nn.ReLU(),
                                     nn.Conv2d(16,8,3, padding=1),
                                     nn.AvgPool2d(2),
                                     nn.ReLU(),
                                     nn.Conv2d(8,4,3, padding=1),
                                     nn.ReLU())
        
        self.decoder = nn.Sequential(nn.Conv2d(4,8,3, padding=1),
                                     nn.Upsample(scale_factor=2),
                                     nn.ReLU(),
                                     nn.Conv2d(8,16,3, padding=1),
                                     nn.Upsample(scale_factor=2),
                                     nn.ReLU(),
                                     nn.Conv2d(16,1,3, padding=1))
        
        #la loss che utilizzeremo per il training
        self.criterion = nn.MSELoss()
        
    def forward(self, x):
        code = self.encoder(x)
        reconstructed = self.decoder(code)
        #restituiamo sia il codice che l'output ricostruito
        return code, reconstructed
    
    # questo metodo definisce l'optimizer
    def configure_optimizers(self):
        optimizer = SGD(self.parameters(), lr=0.01, momentum=0.99) 
        return optimizer
    
    # questo metodo definisce come effettuare ogni singolo step di training
    def training_step(self, train_batch, batch_idx):
        x, _ = train_batch
        _, reconstructed = self.forward(x)
        loss = self.criterion(x, reconstructed)
        
        self.training_step_outputs.extend(reconstructed)
        self.training_step_targets.extend(x)

        self.log('train/loss', loss)
        return loss
    
    def on_train_epoch_end(self):
        self.training_step_outputs = self.training_step_outputs[0].view(-1,1, 28, 28)[:50,...]
        self.training_step_targets = self.training_step_targets[0].view(-1,1, 28, 28)[:50,...]
        self.logger.experiment.add_image('input_images', make_grid(self.training_step_targets, nrow=10, normalize=True),self.global_step)
        self.logger.experiment.add_image('generated_images', make_grid(self.training_step_outputs, nrow=10, normalize=True),self.global_step)

        self.training_step_outputs = []
        self.training_step_targets = []
    
    # questo metodo definisce come effettuare ogni singolo step di validation
    def validation_step(self, val_batch, batch_idx):
        x, _ = val_batch
        _, reconstructed = self.forward(x)
        loss = self.criterion(x, reconstructed)

        self.val_step_outputs.extend(reconstructed)
        self.val_step_targets.extend(x)
        
        self.log('val/loss', loss)
        if batch_idx==0:
            return {'inputs':x, 'outputs':reconstructed}
        
    def on_validation_epoch_end(self):
        self.val_step_outputs = self.val_step_outputs[0].view(-1,1,28, 28)[:50,...]
        self.val_step_targets = self.val_step_targets[0].view(-1,1, 28, 28)[:50,...]
        
        self.logger.experiment.add_image('input_images', make_grid(self.val_step_targets, nrow=10, normalize=True),self.global_step)
        self.logger.experiment.add_image('generated_images', make_grid(self.val_step_outputs, nrow=10, normalize=True),self.global_step)

        self.val_step_outputs = []
        self.val_step_targets = []
        

logger = TensorBoardLogger("tb_logs", name="autoencoder")
    
mnist_conv_autoencoder = conv_autoencoder()
trainer = pl.Trainer(max_epochs=NUM_EPOCHS, accelerator="gpu", devices=1, logger=logger) 
trainer.fit(mnist_conv_autoencoder, mnist_train_loader, mnist_test_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name      | Type       | Params
-----------------------------------------
0 | encoder   | Sequential | 1.6 K 
1 | decoder   | Sequential | 1.6 K 
2 | criterion | MSELoss    | 0     
-----------------------------------------
3.2 K     Trainable params
0         Non-trainable params
3.2 K     Total params
0.013     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Epoch 0:  80%|███████▉  | 47/59 [00:03<00:00, 12.69it/s, v_num=4]

/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:52: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")
